In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd

In [0]:
model_uri = "models:/Readmission_XGBoost_Model@Champion"

model = mlflow.pyfunc.load_model(model_uri)

In [0]:
gold_df = spark.table("db_healthcare_kl.gold.ml_readmission_features")

In [0]:
from pyspark.sql.functions import max

latest_time = (
    gold_df
    .select(max("processed_timestamp"))
    .collect()[0][0]
)

latest_time

In [0]:
new_patients = gold_df.filter(
    gold_df.processed_timestamp == latest_time
)

In [0]:
feature_df = new_patients.drop(
    "person_id",
    "processed_timestamp",
    "readmission_30day_flag"
)

In [0]:
pdf = feature_df.toPandas()

In [0]:
predictions = model.predict(pdf)

In [0]:
result = new_patients.toPandas()

result["prediction"] = predictions

In [0]:
prediction_df = spark.createDataFrame(result)

prediction_df.write \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.gold.readmission_predictions")